# Trajectory and velocity analysis of scRNAseq COLON data 

## Part 1. Data load and quality control

### Healthy dataset

#### 0. Imports and settings

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
import scrublet as scr
import doubletdetection

#### 1. Load data

All files are in the `.loom` format. They have been preprcoessed with *cellranger* (demultiplexing, alignment and counting) and *velocyto* (annotation of spliced/unspliced counts). 

After loading in the file, we have to remove some of the unnecssary data fields generated by previous processing. What we want in the end is just the matrix of raw counts, and matrices of spliced and unspliced reads.

In [ ]:
sample_name = "healthy0"
file = "data/{}.loom".format(sample_name)
adata0 = scv.read_loom(file, sparse=True, cleanup=True)
adata0.var_names_make_unique()
# Delete unnecessary data
del adata0.obs['Clusters']
del adata0.obs['_X']
del adata0.obs['_Y']
del adata0.var['Accession']
del adata0.var['Chromosome']
del adata0.var['End']
del adata0.var['Start']
del adata0.var['Strand']
del adata0.layers['ambiguous']

sample_name = "healthy1"
file = "data/{}.loom".format(sample_name)
adata1 = scv.read_loom(file, sparse=True, cleanup=True)
adata1.var_names_make_unique()
del adata1.obs['Clusters']
del adata1.obs['_X']
del adata1.obs['_Y']
del adata1.var['Accession']
del adata1.var['Chromosome']
del adata1.var['End']
del adata1.var['Start']
del adata1.var['Strand']
del adata1.layers['ambiguous']

sample_name = "healthy7"
file = "data/{}.loom".format(sample_name)
adata7 = scv.read_loom(file, sparse=True, cleanup=True)
adata7.var_names_make_unique()
del adata7.obs['Clusters']
del adata7.obs['_X']
del adata7.obs['_Y']
del adata7.var['Accession']
del adata7.var['Chromosome']
del adata7.var['End']
del adata7.var['Start']
del adata7.var['Strand']
del adata7.layers['ambiguous']

sample_name = "healthy8"
file = "data/{}.loom".format(sample_name)
adata8 = scv.read_loom(file, sparse=True, cleanup=True)
adata8.var_names_make_unique()
del adata8.obs['Clusters']
del adata8.obs['_X']
del adata8.obs['_Y']
del adata8.var['Accession']
del adata8.var['Chromosome']
del adata8.var['End']
del adata8.var['Start']
del adata8.var['Strand']
del adata8.layers['ambiguous']

#### 2. Quality control

Before we can analyse the samples properly we need to filter the cells and genes based on their quality. We start by joining the samples together and calculating some standard quality metrics. 

In [ ]:
# Concatenate the files to calculate joint QC metrics
adata_qc = adata0.concatenate(adata1, adata7, adata8, 
                              batch_key='Sample', batch_categories=['healthy0', 'healthy1', 'healthy7', 'healthy8'])

# Calculate QC covariates
adata_qc.obs['n_counts'] = adata_qc.X.sum(1)
adata_qc.obs['n_spliced'] = adata_qc.layers['spliced'].sum(1)
adata_qc.obs['n_unspliced'] = adata_qc.layers['unspliced'].sum(1)
adata_qc.obs['n_genes'] = (adata_qc.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata_qc.var_names.str.startswith('MT-')
adata_qc.obs['percent_mito'] = np.sum(
    adata_qc[:, mito_genes].X, axis=1) / np.sum(adata_qc.X, axis=1)
ribo_genes = adata_qc.var_names.str.startswith('RP')
adata_qc.obs['percent_ribo'] = np.sum(
    adata_qc[:, ribo_genes].X, axis=1) / np.sum(adata_qc.X, axis=1)
adata_qc.obs['percent_MALAT1'] = np.sum(
    adata_qc[:, 'MALAT1'].X, axis=1) / np.sum(adata_qc.X, axis=1)

# Initial plotting settings for more detailed scatter and dist plots.
#scv.settings.set_figure_params('scvelo', dpi=150, vector_friendly=False)

sc.pl.violin(adata_qc, ['n_genes', 'n_counts'], jitter=0.4, groupby='Sample')
sc.pl.violin(adata_qc, ['percent_mito', 'percent_ribo'], jitter=0.4, groupby='Sample')
sc.pl.violin(adata_qc, ['n_spliced', 'n_unspliced'], jitter=0.4, groupby='Sample')

As can be seen in the figure above, there are significant differences in the distributions of quality metrics in the four samples. This is why filtering based on those metrics is done separately for each of the samples.

##### 2.1 healthy0

In [ ]:
# Calculate QC covariates
adata0.obs['n_counts'] = adata0.X.sum(1)
adata0.obs['log_counts'] = np.log(adata0.obs['n_counts'])
adata0.obs['n_spliced'] = adata0.layers['spliced'].sum(1)
adata0.obs['n_unspliced'] = adata0.layers['unspliced'].sum(1)
adata0.obs['n_genes'] = (adata0.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata0.var_names.str.startswith('MT-')
adata0.obs['percent_mito'] = np.sum(
    adata0[:, mito_genes].X, axis=1) / np.sum(adata0.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata0.var_names.str.startswith('RP')
adata0.obs['percent_ribo'] = np.sum(
    adata0[:, ribo_genes].X, axis=1) / np.sum(adata0.X, axis=1)

sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.scatter(adata0, 'n_counts', 'n_genes', color='percent_mito', title='Percent mitochondrial (colour)', save='qc_healhty')
sc.pl.scatter(adata0, 'n_spliced', 'n_unspliced', save='spliced_healhty')

There is a strong correlation between the number of counts, genes and the mitochondrial content, however, it is still important to look at those statistics jointly. There are many outliers (high number of counts, low number of genes). Specifically, we can see cells with many genes (likely not dead) that have a high mitochondrial content (around 50%). This means we cannot simply exclude all the cells with more than 5% (as done in other studies). 

Here we use the tooll called scrublet to identify doublets - two cells barcoded togheter. Especially important when we are trying too identify cells transitioning between celltypes. We use scrublet to assign doublet score (probability of being a doublet). Actual doublets are detected with the Doublet Detection tool, which is more sofisticated. Uses a classsifier and does not need us to supply a cutoff value for being a doublet. 

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata0.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
scrub.plot_histogram()

In [ ]:
scrub.set_embedding('UMAP', scr.get_umap(scrub.manifold_obs_, 10, min_dist=0.3))
scrub.plot_embedding('UMAP', order_points=True)

In [ ]:
adata0.obs['doublet_score'] = scrub.doublet_scores_obs_

Another tool to annotate doublets is Doublet Detection. More sophisticated, we use it to actually call doublets.

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata0.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f = doubletdetection.plot.convergence(clf, show=True, p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f2, umap_coords = doubletdetection.plot.umap_plot(adata0.X, doublets, random_state=1, show=True)

In [ ]:
adata0.obs['doublet'] = list(map(str, map(int, doublets)))

We look at distribution plots of **count** statistics to set tresholds for **cell filtering**. 

In [ ]:
#Thresholding decision: spliced counts
ax=sb.distplot(adata0.obs['n_spliced'], kde=False)
plt.show()

ax=sb.distplot(adata0.obs['n_spliced'][adata0.obs['n_spliced']<3500], kde=False, bins=60)
plt.show()

ax=sb.distplot(adata0.obs['n_spliced'][adata0.obs['n_spliced']>15000], kde=False, bins=60)
plt.show()

Let's look at unspliced counts.

In [ ]:
#Thresholding decision: unspliced counts
ax=sb.distplot(adata0.obs['n_unspliced'], kde=False)
plt.show()

ax=sb.distplot(adata0.obs['n_unspliced'][adata0.obs['n_unspliced']<2000], kde=False, bins=60)
plt.show()

We look at distribution plots of **gene** statistics to set tresholds for **cell filtering**. 

In [ ]:
#Thresholding decision: genes
ax=sb.distplot(adata0.obs['n_genes'], kde=False, bins=60)
plt.show()
ax=sb.distplot(adata0.obs['n_genes'][adata0.obs['n_genes']<1000], kde=False, bins=60)
plt.show()

By visually tracing a Gaussian around the population we estimate the treshloding limits.

In [ ]:
print("Before filtering cells:\n", adata0, "\n")

sc.pp.filter_cells(adata0, min_counts=1250)
sc.pp.filter_cells(adata0, max_counts=18750)
adata0 = adata0[adata0.obs['n_unspliced']>600]
sc.pp.filter_cells(adata0, min_genes=250)

print("After filtering cells:\n", adata0)

The distribution of genes and counts is now much more balanced.

In [ ]:
sc.pl.violin(adata0, ['n_genes', 'n_counts', 'percent_mito'],
             jitter=0.4, multi_panel=True)

Let's now focus on the the **mitochondrial content**.

In [ ]:
sc.pl.scatter(adata0, 'n_genes', 'percent_mito')

Now we look at distribution of the mitochondrial content and compare it with the number of genes. In general, the content is quite high across all the cells in the sample and since we want to capture the majority of the population, **we set the limit to 50%**. That is a very significant amount, however, those cells still have a lot of genes, which suggests that they are alive and biologically relevant. Another explanation could be that the high content is due to a technical effect and affects all the cells, meaning it does not accurately represents variation in our dataset. 

In any case, we should be permissive with our treshold.

In [ ]:
adata0 = adata0[adata0.obs['percent_mito'] < 0.3, :]
sc.pl.scatter(adata0, 'n_counts', 'n_genes', color='percent_mito', title='Percent mitochondrial genes')
print(adata0)

Next we filter out the remaining doublet cells. We filter based on the classification from the Doublet Detection method. We compare the agreement between the methods by then visualising the score from the scrublet method. We can see that some cells that scored very highly in scrublet are not classified as doublets by the other software. Those may be false negatives, so we filter them out as well to be sure we don't have any doublets. This is done at the cost of filtering out some false positives. However, the assumption is that the cell populations we are interested in are common enough that they would not be affected. 

In [ ]:
sc.pl.scatter(adata0, 'n_genes', 'doublet_score', color='doublet', palette=['gray', 'orange'], legend_loc='none', title='', save='healthy_doublet')

In [ ]:
adata0 = adata0[adata0.obs['doublet'] != '1']
adata0 = adata0[adata0.obs['doublet_score'] < 0.3]

Finally, we can filter out genes. This is done mostly to decrease the size of the object and speed up calculations later on by getting rid off uninformative genes. We do it per sample, so that we remove sample specific genes later on. We keep genes that have counts in at least 5 cells. Filtering out genes also helps speed up subsequent calculations.

In [ ]:
sc.pl.highest_expr_genes(adata0, n_top=20)
print(adata0)

##### 2.2 healthy1

The analysis and its reasoning is analogous to the previous one. See section **2.1** for describtion.

In [ ]:
# Calculate QC covariates
adata1.obs['n_counts'] = adata1.X.sum(1)
adata1.obs['log_counts'] = np.log(adata1.obs['n_counts'])
adata1.obs['n_genes'] = (adata1.X > 0).sum(1)
adata1.obs['n_spliced'] = adata1.layers['spliced'].sum(1)
adata1.obs['n_unspliced'] = adata1.layers['unspliced'].sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata1.var_names.str.startswith('MT-')
adata1.obs['percent_mito'] = np.sum(
    adata1[:, mito_genes].X, axis=1) / np.sum(adata1.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata1.var_names.str.startswith('RP')
adata1.obs['percent_ribo'] = np.sum(
    adata1[:, ribo_genes].X, axis=1) / np.sum(adata1.X, axis=1)

sc.pl.scatter(adata1, 'n_counts', 'n_genes', color='percent_mito', title='Percent mitochondrial genes')
sc.pl.scatter(adata1, 'n_spliced', 'n_unspliced')

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata1.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
scrub.plot_histogram()

In [ ]:
scrub.set_embedding('UMAP', scr.get_umap(scrub.manifold_obs_, 10, min_dist=0.3))
scrub.plot_embedding('UMAP', order_points=True)

In [ ]:
adata1.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata1.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f = doubletdetection.plot.convergence(clf, show=True, p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f2, umap_coords = doubletdetection.plot.umap_plot(adata1.X, doublets, random_state=1, show=True)

In [ ]:
adata1.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
#Thresholding decision: spliced counts
ax=sb.distplot(adata1.obs['n_spliced'], kde=False)
plt.show()

ax=sb.distplot(adata1.obs['n_spliced'][adata1.obs['n_spliced']<4000], kde=False, bins=60)
plt.show()

ax=sb.distplot(adata1.obs['n_spliced'][adata1.obs['n_spliced']>20000], kde=False, bins=60)
plt.show()

Let's look at unspliced counts.

In [ ]:
#Thresholding decision: unspliced counts
ax=sb.distplot(adata1.obs['n_unspliced'], kde=False)
plt.show()

ax=sb.distplot(adata1.obs['n_unspliced'][adata1.obs['n_unspliced']<1750], kde=False, bins=60)
plt.show()

In [ ]:
#Thresholding decision: genes
ax=sb.distplot(adata1.obs['n_genes'], kde=False, bins=60)
plt.show()
ax=sb.distplot(adata1.obs['n_genes'][adata1.obs['n_genes']<1750], kde=False, bins=60)
plt.show()

In [ ]:
print("Before filtering cells:\n", adata1, "\n")

sc.pp.filter_cells(adata1, min_counts=2100)
sc.pp.filter_cells(adata1, max_counts=26000)
adata1 = adata1[adata1.obs['n_unspliced']>700]
sc.pp.filter_cells(adata1, min_genes=900)

print("After filtering cells:\n", adata1)

In [ ]:
sc.pl.violin(adata1, ['n_genes', 'n_counts', 'percent_mito'],
             jitter=0.4, multi_panel=True)

In [ ]:
sc.pl.scatter(adata1, 'n_genes', 'percent_mito')

In [ ]:
adata1 = adata1[adata1.obs['percent_mito'] < 0.3, :]
sc.pl.scatter(adata1, 'n_counts', 'n_genes', color='percent_mito', title='Percent mitochondrial genes')
print(adata1)

In [ ]:
sc.pl.scatter(adata1, 'n_genes', 'doublet_score', color='doublet', palette=['gray', 'orange'], legend_loc='none', title='Doublet')
adata1 = adata1[adata1.obs['doublet'] != '1']
adata1 = adata1[adata1.obs['doublet_score'] < 0.3]

In [ ]:
sc.pl.highest_expr_genes(adata1, n_top=20)
print(adata1)

##### 2.3 healthy7

The analysis and its reasoning is analogous to the previous one. See section **2.1** for describtion.

In [ ]:
# Calculate QC covariates
adata7.obs['n_counts'] = adata7.X.sum(1)
adata7.obs['log_counts'] = np.log(adata7.obs['n_counts'])
adata7.obs['n_genes'] = (adata7.X > 0).sum(1)
adata7.obs['n_spliced'] = adata7.layers['spliced'].sum(1)
adata7.obs['n_unspliced'] = adata7.layers['unspliced'].sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata7.var_names.str.startswith('MT-')
adata7.obs['percent_mito'] = np.sum(
    adata7[:, mito_genes].X, axis=1) / np.sum(adata7.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata7.var_names.str.startswith('RP')
adata7.obs['percent_ribo'] = np.sum(
    adata7[:, ribo_genes].X, axis=1) / np.sum(adata7.X, axis=1)

sc.pl.scatter(adata7, 'n_counts', 'n_genes', color='percent_mito', title='Percent mitochondrial genes')
sc.pl.scatter(adata7, 'n_spliced', 'n_unspliced')

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata7.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
scrub.plot_histogram()

In [ ]:
scrub.set_embedding('UMAP', scr.get_umap(scrub.manifold_obs_, 10, min_dist=0.3))
scrub.plot_embedding('UMAP', order_points=True)

In [ ]:
adata7.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata7.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f = doubletdetection.plot.convergence(clf, show=True, p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f2, umap_coords = doubletdetection.plot.umap_plot(adata7.X, doublets, random_state=1, show=True)

In [ ]:
adata7.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
#Thresholding decision: spliced counts
ax=sb.distplot(adata7.obs['n_spliced'], kde=False)
plt.show()

ax=sb.distplot(adata7.obs['n_spliced'][adata7.obs['n_spliced']<2500], kde=False, bins=60)
plt.show()

ax=sb.distplot(adata7.obs['n_spliced'][adata7.obs['n_spliced']>12500], kde=False, bins=60)
plt.show()

Let's look at unspliced counts.

In [ ]:
#Thresholding decision: unspliced counts
ax=sb.distplot(adata7.obs['n_unspliced'], kde=False)
plt.show()

ax=sb.distplot(adata7.obs['n_unspliced'][adata7.obs['n_unspliced']<1500], kde=False, bins=60)
plt.show()

In [ ]:
#Thresholding decision: genes
ax=sb.distplot(adata7.obs['n_genes'], kde=False, bins=60)
plt.show()
ax=sb.distplot(adata7.obs['n_genes'][adata7.obs['n_genes']<1000], kde=False, bins=60)
plt.show()

In [ ]:
print("Before filtering cells:\n", adata7, "\n")

sc.pp.filter_cells(adata7, min_counts=1600)
sc.pp.filter_cells(adata7, max_counts=18000)
adata7 = adata7[adata7.obs['n_unspliced']>750]
sc.pp.filter_cells(adata7, min_genes=500)

print("After filtering cells:\n", adata7)

In [ ]:
sc.pl.violin(adata7, ['n_genes', 'n_counts', 'percent_mito'],
             jitter=0.4, multi_panel=True)

In [ ]:
sc.pl.scatter(adata7, 'n_genes', 'percent_mito')

In [ ]:
adata7 = adata7[adata7.obs['percent_mito'] < 0.3, :]
sc.pl.scatter(adata7, 'n_counts', 'n_genes', color='percent_mito', title='Percent mitochondrial genes)')
print(adata7)

In [ ]:
sc.pl.scatter(adata7, 'n_genes', 'doublet_score', color='doublet', palette=['gray', 'orange'], legend_loc='none', title='Doublet')
adata7 = adata7[adata7.obs['doublet'] != '1']
adata7 = adata7[adata7.obs['doublet_score'] < 0.3]

In [ ]:
sc.pl.highest_expr_genes(adata7, n_top=20)
print(adata7)

##### 2.4 healthy8

The analysis and its reasoning is analogous to the previous one. See section **2.1** for describtion.

In [ ]:
# Calculate QC covariates
adata8.obs['n_counts'] = adata8.X.sum(1)
adata8.obs['log_counts'] = np.log(adata8.obs['n_counts'])
adata8.obs['n_genes'] = (adata8.X > 0).sum(1)
adata8.obs['n_spliced'] = adata8.layers['spliced'].sum(1)
adata8.obs['n_unspliced'] = adata8.layers['unspliced'].sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata8.var_names.str.startswith('MT-')
adata8.obs['percent_mito'] = np.sum(
    adata8[:, mito_genes].X, axis=1) / np.sum(adata8.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata8.var_names.str.startswith('RP')
adata8.obs['percent_ribo'] = np.sum(
    adata8[:, ribo_genes].X, axis=1) / np.sum(adata8.X, axis=1)

sc.pl.scatter(adata8, 'n_counts', 'n_genes', color='percent_mito', title='Percent mitochondrial genes')
sc.pl.scatter(adata8, 'n_spliced', 'n_unspliced')

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata8.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
scrub.plot_histogram()

In [ ]:
scrub.set_embedding('UMAP', scr.get_umap(scrub.manifold_obs_, 10, min_dist=0.3))
scrub.plot_embedding('UMAP', order_points=True)

In [ ]:
adata8.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata8.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f = doubletdetection.plot.convergence(clf, show=True, p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
f2, umap_coords = doubletdetection.plot.umap_plot(adata8.X, doublets, random_state=1, show=True)

In [ ]:
adata8.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
#Thresholding decision: spliced counts
ax=sb.distplot(adata8.obs['n_spliced'], kde=False)
plt.show()

ax=sb.distplot(adata8.obs['n_spliced'][adata8.obs['n_spliced']<2500], kde=False, bins=60)
plt.show()

ax=sb.distplot(adata8.obs['n_spliced'][adata8.obs['n_spliced']>10000], kde=False, bins=60)
plt.show()

Let's look at unspliced counts.

In [ ]:
#Thresholding decision: unspliced counts
ax=sb.distplot(adata8.obs['n_unspliced'], kde=False)
plt.show()

ax=sb.distplot(adata8.obs['n_unspliced'][adata8.obs['n_unspliced']<1500], kde=False, bins=60)
plt.show()

In [ ]:
#Thresholding decision: genes
ax=sb.distplot(adata8.obs['n_genes'], kde=False, bins=60)
plt.show()
ax=sb.distplot(adata8.obs['n_genes'][adata8.obs['n_genes']<1000], kde=False, bins=60)
plt.show()

In [ ]:
print("Before filtering cells:\n", adata8, "\n")

sc.pp.filter_cells(adata8, min_counts=1600)
sc.pp.filter_cells(adata8, max_counts=16000)
adata8 = adata8[adata8.obs['n_unspliced']>800]
sc.pp.filter_cells(adata8, min_genes=500)

print("After filtering cells:\n", adata8)

In [ ]:
sc.pl.violin(adata8, ['n_genes', 'n_counts', 'percent_mito'],
             jitter=0.4, multi_panel=True)

In [ ]:
sc.pl.scatter(adata8, 'n_genes', 'percent_mito')

In [ ]:
adata8 = adata8[adata8.obs['percent_mito'] < 0.3, :]
sc.pl.scatter(adata8, 'n_counts', 'n_genes', color='percent_mito', title='Percent mitochondrial genes')
print(adata8)

In [ ]:
sc.pl.scatter(adata8, 'n_genes', 'doublet_score', color='doublet', palette=['gray', 'orange'], legend_loc='none', title='Doublet')
adata8 = adata8[adata8.obs['doublet'] != '1']
adata8 = adata8[adata8.obs['doublet_score'] < 0.3]

In [ ]:
sc.pl.highest_expr_genes(adata8, n_top=20)
print(adata8)

#### 3. Merge samples

For the rest of the analysis we concatenate the samples into one object `adata`. First, we normalise the counts per cell (CPM normalisation) and log transform them. At this point we save the expression matrix to the `.raw` attribute of our object. It is important to keep the "raw" values for DE analysis or visulaising markers expression.   

There's the option to normalise all cells to 1e6 counts per cell (CPM normalisation). However, this would mask differences between cells of different sizes. Instead we normalise "to the median of total counts for observations (cells) before normalization". Layers (spliced, unspliced) are also normalised to the median of counts (`adata.X`).

In [ ]:
adata = adata0.concatenate(adata1, adata7, adata8,
                           batch_key='Sample', 
                           batch_categories=['healthy0', 'healthy1', 'healthy7', 'healthy8'])
# Apply workaround necessary due to bugs in code
adata.layers['spliced'] = adata.layers['spliced'].astype(float)
adata.layers['unspliced'] = adata.layers['unspliced'].astype(float)

scv.pp.filter_genes(adata, min_cells=5)
scv.pp.filter_genes(adata, min_counts=20)

adata

#### 4.Save data for the next step 

Saving data with a unique name after each step will prevent overwriting. That means you will always be able to easily come back to a previous step in the pipeline without the need to run everything over again. This is good for the first time trying this pipeline in case something breaks along the way. However, in the future you may prefer to save to a single file in order to reduce disk space usage. 

In [ ]:
adata.write("healthy_qc.h5ad")